This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision transformers --upgrade -q

## Image segmentation

### Computer vision tasks

#### Types of image segmentation

### Training a segmentation model from scratch

#### Downloading a segmentation dataset

In [0]:
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
!tar -xf images.tar.gz
!tar -xf annotations.tar.gz

In [0]:
import pathlib

input_dir = pathlib.Path("images")
target_dir = pathlib.Path("annotations/trimaps")

input_img_paths = sorted(input_dir.glob("*.jpg"))
target_paths = sorted(target_dir.glob("[!.]*.png"))

In [ ]:
import matplotlib.pyplot as plt
# PyTorch: there's no keras.utils.load_img; use PIL to open images.
from PIL import Image

plt.axis("off")
plt.imshow(Image.open(input_img_paths[9]))

In [ ]:
def display_target(target_array):
    normalized_array = (target_array.astype("uint8") - 1) * 127
    plt.axis("off")
    plt.imshow(normalized_array[:, :, 0])

# PyTorch: load the grayscale trimap with PIL and convert to a (H, W, 1) array.
img = np.array(Image.open(target_paths[9]).convert("L"))[:, :, None]
display_target(img)

In [ ]:
import numpy as np
import random

img_size = (200, 200)
num_imgs = len(input_img_paths)

random.Random(1337).shuffle(input_img_paths)
random.Random(1337).shuffle(target_paths)

# PyTorch: replace keras load_img/img_to_array with PIL. We keep the arrays in
# NHWC (channels-last) so the display code below is unchanged; the Dataset will
# permute to NCHW for the model.
def path_to_input_image(path):
    img = Image.open(path).convert("RGB").resize(img_size[::-1])
    return np.array(img, dtype="float32")

def path_to_target(path):
    img = Image.open(path).convert("L").resize(img_size[::-1])
    img = np.array(img, dtype="uint8")[:, :, None]
    img = img.astype("uint8") - 1
    return img

input_imgs = np.zeros((num_imgs,) + img_size + (3,), dtype="float32")
targets = np.zeros((num_imgs,) + img_size + (1,), dtype="uint8")
for i in range(num_imgs):
    input_imgs[i] = path_to_input_image(input_img_paths[i])
    targets[i] = path_to_target(target_paths[i])

In [0]:
num_val_samples = 1000
train_input_imgs = input_imgs[:-num_val_samples]
train_targets = targets[:-num_val_samples]
val_input_imgs = input_imgs[-num_val_samples:]
val_targets = targets[-num_val_samples:]

#### Building and training the segmentation model

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

# PyTorch: the Keras Functional model becomes an nn.Module subclass. Conv2D ->
# nn.Conv2d, Conv2DTranspose -> nn.ConvTranspose2d. Inputs are NCHW (not NHWC),
# and "same" padding with kernel 3 means padding=1. For the strided transposed
# convs we add output_padding=1 so the spatial size doubles exactly. Rescaling
# (1/255) is folded into the forward pass. We drop the final softmax because
# nn.CrossEntropyLoss operates on raw logits.
class SegmentationModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, stride=2, padding=1)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2, padding=1)
        self.conv4 = nn.Conv2d(128, 128, 3, padding=1)
        self.conv5 = nn.Conv2d(128, 256, 3, stride=2, padding=1)
        self.conv6 = nn.Conv2d(256, 256, 3, padding=1)

        self.up1 = nn.ConvTranspose2d(256, 256, 3, padding=1)
        self.up2 = nn.ConvTranspose2d(256, 256, 3, stride=2, padding=1, output_padding=1)
        self.up3 = nn.ConvTranspose2d(256, 128, 3, padding=1)
        self.up4 = nn.ConvTranspose2d(128, 128, 3, stride=2, padding=1, output_padding=1)
        self.up5 = nn.ConvTranspose2d(128, 64, 3, padding=1)
        self.up6 = nn.ConvTranspose2d(64, 64, 3, stride=2, padding=1, output_padding=1)

        self.classifier = nn.Conv2d(64, num_classes, 3, padding=1)

    def forward(self, x):
        x = x / 255.0
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = F.relu(self.conv6(x))

        x = F.relu(self.up1(x))
        x = F.relu(self.up2(x))
        x = F.relu(self.up3(x))
        x = F.relu(self.up4(x))
        x = F.relu(self.up5(x))
        x = F.relu(self.up6(x))

        return self.classifier(x)

def get_model(img_size, num_classes):
    return SegmentationModel(num_classes)

model = get_model(img_size=img_size, num_classes=3)

In [ ]:
# PyTorch: the original Keras IoU metric note doesn't apply here. We compute a
# simple foreground IoU by hand in the training loop below instead.

In [ ]:
# PyTorch: there's no keras.metrics.IoU. We define foreground IoU (class 0)
# directly: intersection over union between predicted and true foreground pixels.
def foreground_iou(preds, targets, class_id=0):
    pred_cls = preds == class_id
    true_cls = targets == class_id
    intersection = (pred_cls & true_cls).sum().item()
    union = (pred_cls | true_cls).sum().item()
    return intersection / union if union > 0 else 0.0

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# PyTorch: model.compile becomes an explicit optimizer + loss. nn.CrossEntropyLoss
# is the equivalent of sparse_categorical_crossentropy over pixels: it takes
# (N, C, H, W) logits and (N, H, W) long targets.
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.CrossEntropyLoss()

# PyTorch: build NCHW tensors. Images permute HWC -> CHW; targets drop the
# channel dim to become (N, H, W) integer class maps.
def make_loader(imgs, tgts, batch_size, shuffle):
    x = torch.from_numpy(imgs).permute(0, 3, 1, 2).contiguous()
    y = torch.from_numpy(tgts[..., 0].astype("int64"))
    return DataLoader(TensorDataset(x, y), batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(train_input_imgs, train_targets, 64, shuffle=True)
val_loader = make_loader(val_input_imgs, val_targets, 64, shuffle=False)

# PyTorch: model.fit becomes a manual loop. We track loss/val_loss in a history
# dict (mirroring Keras' history.history) and save the best model by val loss
# (the equivalent of ModelCheckpoint(save_best_only=True)).
history = {"history": {"loss": [], "val_loss": [], "foreground_iou": [], "val_foreground_iou": []}}
best_val_loss = float("inf")
epochs = 50
for epoch in range(epochs):
    model.train()
    train_loss, train_iou, n = 0.0, 0.0, 0
    for inputs, tgts in train_loader:
        inputs, tgts = inputs.to(device), tgts.to(device)
        preds = model(inputs)
        loss = loss_fn(preds, tgts)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * inputs.size(0)
        train_iou += foreground_iou(preds.argmax(1), tgts) * inputs.size(0)
        n += inputs.size(0)
    train_loss /= n
    train_iou /= n

    model.eval()
    val_loss, val_iou, vn = 0.0, 0.0, 0
    with torch.no_grad():
        for inputs, tgts in val_loader:
            inputs, tgts = inputs.to(device), tgts.to(device)
            preds = model(inputs)
            val_loss += loss_fn(preds, tgts).item() * inputs.size(0)
            val_iou += foreground_iou(preds.argmax(1), tgts) * inputs.size(0)
            vn += inputs.size(0)
    val_loss /= vn
    val_iou /= vn

    history["history"]["loss"].append(train_loss)
    history["history"]["val_loss"].append(val_loss)
    history["history"]["foreground_iou"].append(train_iou)
    history["history"]["val_foreground_iou"].append(val_iou)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "oxford_segmentation.pt")
    print(f"Epoch {epoch + 1}/{epochs}: loss {train_loss:.4f} - val_loss {val_loss:.4f}")

In [ ]:
epochs = range(1, len(history["history"]["loss"]) + 1)
loss = history["history"]["loss"]
val_loss = history["history"]["val_loss"]
plt.figure()
plt.plot(epochs, loss, "r--", label="Training loss")
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.legend()

In [ ]:
# PyTorch: reload the best weights with load_state_dict (instead of load_model).
model.load_state_dict(torch.load("oxford_segmentation.pt"))

i = 4
test_image = val_input_imgs[i]
plt.axis("off")
# PyTorch: array_to_img has no direct equivalent; show the uint8 NHWC array.
plt.imshow(test_image.astype("uint8"))

# PyTorch: model.predict becomes eval() + no_grad. We permute the single image to
# NCHW, run the model, and softmax + move back to NHWC to match the original
# (H, W, num_classes) prediction the display code below expects.
model.eval()
with torch.no_grad():
    x = torch.from_numpy(test_image).permute(2, 0, 1).unsqueeze(0).to(device)
    logits = model(x)
    mask = torch.softmax(logits, dim=1)[0].permute(1, 2, 0).cpu().numpy()

def display_mask(pred):
    mask = np.argmax(pred, axis=-1)
    mask *= 127
    plt.axis("off")
    plt.imshow(mask)

display_mask(mask)

### Using a pretrained segmentation model

#### Downloading the Segment Anything Model

In [ ]:
# PyTorch: keras_hub's "sam_huge_sa1b" Segment Anything preset maps to the
# official SAM weights distributed via HuggingFace transformers. We use the
# facebook/sam-vit-huge checkpoint, which is the same SA-1B-trained ViT-Huge SAM.
from transformers import SamModel, SamProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
model = SamModel.from_pretrained("facebook/sam-vit-huge").to(device)
processor = SamProcessor.from_pretrained("facebook/sam-vit-huge")

In [ ]:
# PyTorch: sum the number of elements across all parameters.
sum(p.numel() for p in model.parameters())

#### How Segment Anything works

#### Preparing a test image

In [ ]:
# PyTorch: download the test image with torch.hub and open it with PIL.
import torch
from PIL import Image

path = torch.hub.download_url_to_file(
    "https://s3.amazonaws.com/keras.io/img/book/fruits.jpg", "fruits.jpg"
) or "fruits.jpg"
pil_image = Image.open("fruits.jpg").convert("RGB")
image_array = np.array(pil_image, dtype="float32")

plt.imshow(image_array.astype("uint8"))
plt.axis("off")
plt.show()

In [ ]:
# PyTorch: the SamProcessor handles resizing/padding internally, so for the
# overlay visualizations we just keep the original image. We define a helper that
# resizes a HWC array (image or mask) to image_size while preserving aspect ratio
# via padding, using torchvision functional transforms.
import torchvision.transforms.functional as TF

image_size = (1024, 1024)

def resize_and_pad(x):
    t = torch.from_numpy(np.asarray(x, dtype="float32")).permute(2, 0, 1)
    c, h, w = t.shape
    scale = min(image_size[0] / h, image_size[1] / w)
    new_h, new_w = int(round(h * scale)), int(round(w * scale))
    t = TF.resize(t, [new_h, new_w], antialias=True)
    pad_h, pad_w = image_size[0] - new_h, image_size[1] - new_w
    t = TF.pad(t, [0, 0, pad_w, pad_h])
    return t.permute(1, 2, 0).numpy()

image = resize_and_pad(image_array)

In [ ]:
import matplotlib.pyplot as plt

# PyTorch: replace ops.convert_to_numpy with np.asarray.
def show_image(image, ax):
    ax.imshow(np.asarray(image).astype("uint8"))

def show_mask(mask, ax):
    color = np.array([30 / 255, 144 / 255, 255 / 255, 0.6])
    h, w, _ = mask.shape
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

def show_points(points, ax):
    x, y = points[:, 0], points[:, 1]
    ax.scatter(x, y, c="green", marker="*", s=375, ec="white", lw=1.25)

def show_box(box, ax):
    box = box.reshape(-1)
    x0, y0 = box[0], box[1]
    w, h = box[2] - box[0], box[3] - box[1]
    ax.add_patch(plt.Rectangle((x0, y0), w, h, ec="red", fc="none", lw=2))

#### Prompting the model with a target point

In [0]:
import numpy as np

input_point = np.array([[580, 450]])
input_label = np.array([1])

plt.figure(figsize=(10, 10))
show_image(image, plt.gca())
show_points(input_point, plt.gca())
plt.show()

In [ ]:
# PyTorch: SAM inference via transformers. The SamProcessor takes the PIL image
# plus point prompts (as [[ [[x, y]] ]] nested lists) and produces model inputs;
# the processor then post-processes the low-res logits back to image resolution.
inputs = processor(
    pil_image,
    input_points=[[input_point.tolist()]],
    input_labels=[[input_label.tolist()]],
    return_tensors="pt",
).to(device)
with torch.no_grad():
    sam_out = model(**inputs)
masks = processor.image_processor.post_process_masks(
    sam_out.pred_masks.cpu(),
    inputs["original_sizes"].cpu(),
    inputs["reshaped_input_sizes"].cpu(),
)
# PyTorch: wrap results in a dict shaped like the original Keras outputs:
# "masks" -> (batch, num_masks, H, W), "iou" -> mask quality scores.
outputs = {"masks": masks[0].numpy(), "iou_scores": sam_out.iou_scores.cpu().numpy()}

In [ ]:
outputs["masks"].shape

In [ ]:
# PyTorch: post_process_masks already returns boolean masks at the original image
# resolution, shaped (num_masks, H, W). We pick one, add a channel dim, and resize
# it to the padded image_size to overlay on `image`.
def get_mask(sam_outputs, index=0):
    mask = sam_outputs["masks"][index]
    mask = np.expand_dims(mask.astype("float32"), axis=-1)
    mask = resize_and_pad(mask)
    return mask > 0.0

mask = get_mask(outputs, index=0)

plt.figure(figsize=(10, 10))
show_image(image, plt.gca())
show_mask(mask, plt.gca())
show_points(input_point, plt.gca())
plt.show()

In [ ]:
input_point = np.array([[300, 550]])
input_label = np.array([1])

inputs = processor(
    pil_image,
    input_points=[[input_point.tolist()]],
    input_labels=[[input_label.tolist()]],
    return_tensors="pt",
).to(device)
with torch.no_grad():
    sam_out = model(**inputs)
masks = processor.image_processor.post_process_masks(
    sam_out.pred_masks.cpu(),
    inputs["original_sizes"].cpu(),
    inputs["reshaped_input_sizes"].cpu(),
)
outputs = {"masks": masks[0].numpy(), "iou_scores": sam_out.iou_scores.cpu().numpy()}
mask = get_mask(outputs, index=0)

plt.figure(figsize=(10, 10))
show_image(image, plt.gca())
show_mask(mask, plt.gca())
show_points(input_point, plt.gca())
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 60))
# PyTorch: our outputs["masks"] is (num_masks, H, W), so index the extra masks
# directly (no leading batch axis as in the original Keras outputs).
extra = outputs["masks"][1:]
for i in range(len(extra)):
    show_image(image, axes[i])
    show_points(input_point, axes[i])
    mask = get_mask(outputs, index=i + 1)
    show_mask(mask, axes[i])
    axes[i].set_title(f"Mask {i + 1}", fontsize=16)
    axes[i].axis("off")
plt.show()

#### Prompting the model with a target box

In [0]:
input_box = np.array(
    [
        [520, 180],
        [770, 420],
    ]
)

plt.figure(figsize=(10, 10))
show_image(image, plt.gca())
show_box(input_box, plt.gca())
plt.show()

In [ ]:
# PyTorch: SAM box prompts are passed as input_boxes [x0, y0, x1, y1].
input_boxes = [[[float(input_box[0, 0]), float(input_box[0, 1]),
                 float(input_box[1, 0]), float(input_box[1, 1])]]]
inputs = processor(
    pil_image,
    input_boxes=input_boxes,
    return_tensors="pt",
).to(device)
with torch.no_grad():
    sam_out = model(**inputs)
masks = processor.image_processor.post_process_masks(
    sam_out.pred_masks.cpu(),
    inputs["original_sizes"].cpu(),
    inputs["reshaped_input_sizes"].cpu(),
)
outputs = {"masks": masks[0].numpy(), "iou_scores": sam_out.iou_scores.cpu().numpy()}
mask = get_mask(outputs, 0)
plt.figure(figsize=(10, 10))
show_image(image, plt.gca())
show_mask(mask, plt.gca())
show_box(input_box, plt.gca())
plt.show()